# S02 — Feature Engineering QA

Visual validation of Phase 2 outputs: risk index construction, feature distributions, and data quality checks.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA = ROOT / "data"
full = pd.read_parquet(DATA / "spain_grid_features.parquet")
train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")
fit = joblib.load(DATA / "artifacts" / "risk_index_fit.joblib")

print(f"Full: {len(full):,}  Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}")

## 1. Core Factor Distributions

In [ ]:
factor_cols = ["flexibility_share", "demand_forecast_error", "net_load"]

fig = make_subplots(rows=1, cols=3, subplot_titles=factor_cols)
for i, col in enumerate(factor_cols, 1):
    fig.add_trace(
        go.Histogram(x=train[col], name=col, nbinsx=80, marker_color="steelblue"),
        row=1, col=i,
    )
fig.update_layout(title="Core Factor Distributions (Train)", showlegend=False, height=350)
fig.show()

## 2. PCA Explained Variance & Biplot

In [ ]:
ev = fit.pca.explained_variance_ratio_
fig = go.Figure()
fig.add_trace(go.Bar(x=[f"PC{i+1}" for i in range(3)], y=ev, marker_color="steelblue"))
fig.add_trace(go.Scatter(
    x=[f"PC{i+1}" for i in range(3)], y=np.cumsum(ev),
    mode="lines+markers", name="Cumulative", line=dict(color="firebrick"),
))
fig.update_layout(title="PCA Explained Variance", yaxis_title="Proportion", height=350)
fig.show()

# Biplot: loadings
loadings = fit.pca.components_[:2].T  # (3 factors × 2 PCs)
fig2 = go.Figure()
for i, name in enumerate(factor_cols):
    fig2.add_trace(go.Scatter(
        x=[0, loadings[i, 0]], y=[0, loadings[i, 1]],
        mode="lines+markers+text", text=["", name],
        textposition="top center", name=name,
    ))
fig2.update_layout(title="PCA Biplot (PC1 vs PC2)", xaxis_title="PC1", yaxis_title="PC2",
                   height=400, width=500)
fig2.show()

## 3. Risk Index Time Series with Split Boundaries

In [ ]:
# Daily mean for readability
daily = full["risk_index"].resample("D").mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.index, y=daily.values, mode="lines", name="risk_index (daily mean)",
                         line=dict(width=0.8, color="steelblue")))

for label, ts in [("Train→Val", "2024-01-01"), ("Val→Test", "2024-07-01")]:
    fig.add_vline(x=ts, line_dash="dash", line_color="red")
    fig.add_annotation(x=ts, y=1.05, yref="paper", text=label,
                       showarrow=False, font=dict(color="red"))

fig.update_layout(title="Risk Index Time Series", yaxis_title="risk_index", height=400)
fig.show()

## 4. Risk Index Distribution Overlap Across Splits

In [ ]:
fig = go.Figure()
for label, split, color in [("Train", train, "steelblue"), ("Val", val, "orange"), ("Test", test, "green")]:
    fig.add_trace(go.Histogram(
        x=split["risk_index"], name=label, nbinsx=60,
        marker_color=color, opacity=0.5, histnorm="probability density",
    ))
fig.update_layout(title="Risk Index Distribution by Split", barmode="overlay",
                  xaxis_title="risk_index", height=350)
fig.show()

## 5. Risk Category Counts per Split

In [ ]:
counts = pd.DataFrame({
    label: split["risk_category"].value_counts()
    for label, split in [("Train", train), ("Val", val), ("Test", test)]
}).T
counts = counts[["Low", "Medium", "High"]]
display(counts)

fig = px.bar(counts.reset_index(), x="index", y=["Low", "Medium", "High"],
             barmode="group", title="Risk Category Counts per Split",
             labels={"index": "Split", "value": "Count"})
fig.update_layout(height=350)
fig.show()

## 6. Feature Correlation Heatmap vs. Risk Index

In [ ]:
from grid_risk.features import FEATURE_NAMES

corr_cols = FEATURE_NAMES + ["risk_index"]
corr = train[corr_cols].corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Feature Correlation Matrix (Train)")
fig.update_layout(height=550, width=650)
fig.show()

## 7. Daily-Generation Limitation Check

Within-day variance of `flexibility_share` should be ~0 (daily broadcast), while `net_load` and `demand_forecast_error` should show real hourly variance.

In [ ]:
# Compute within-day std for each factor
daily_std = train.groupby(train.index.date)[
    ["flexibility_share", "demand_forecast_error", "net_load"]
].std()

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["flexibility_share", "demand_forecast_error", "net_load"])
for i, col in enumerate(["flexibility_share", "demand_forecast_error", "net_load"], 1):
    fig.add_trace(
        go.Histogram(x=daily_std[col], nbinsx=60, marker_color="steelblue"),
        row=1, col=i,
    )
fig.update_layout(title="Within-Day Std of Core Factors (Train)",
                  showlegend=False, height=350)
fig.show()

print("Median within-day std:")
print(daily_std.median().to_string())

## Leakage Checks

In [ ]:
# Verify scaler mean/std match train-only statistics
train_factors = train[["flexibility_share", "demand_forecast_error", "net_load"]]
print("Scaler mean (fit):", fit.scaler.mean_)
print("Train mean (calc):", train_factors.mean().values)
print()
print("Scaler std  (fit):", fit.scaler.scale_)
print("Train std  (calc):", train_factors.std(ddof=0).values)
print()

# Verify lag alignment: risk_index_lag_24h at t == risk_index at t-24
sample_t = train.index[200]
lag_val = train.loc[sample_t, "risk_index_lag_24h"]
actual_val = full.loc[sample_t - pd.Timedelta(hours=24), "risk_index"]
print(f"Lag check at {sample_t}:")
print(f"  risk_index_lag_24h = {lag_val:.6f}")
print(f"  risk_index[t-24h]  = {actual_val:.6f}")
print(f"  Match: {np.isclose(lag_val, actual_val)}")